# Invoice OCR → JSON → Generic HS RAG

Works across **unknown invoices, unknown products, unknown countries**.

Nothing about a vendor, SKU family, or destination is hardcoded.

**Per PDF the system only uses**
1. Text extracted from that PDF.
2. A public international HS heading catalog (downloaded at runtime, ~6k HS-2/4/6 titles).
3. Optional country-specific schedule files *you* drop in, if you have them.

**What it will not do**
- Guess a national 8/10-digit code for a country whose official schedule is not loaded.
- Treat a part number as an HS code.
- Use Google snippets as a tariff schedule.

Default output is **international HS-6** plus `needs_review`. National digits are filled only when a matching official national file exists for the destination country found on that invoice.


## STEP 1 — Environment setup

Run these cells first. Pin versions only if you are reproducing the original Colab A100 box. On a fresh runtime, unpinned current wheels are usually fine.


In [1]:
!pip install torch==2.10.0 torchvision==0.25.0 transformers==4.57.1 Pillow==12.1.1 matplotlib==3.10.8 einops==0.8.2 addict==2.4.0 easydict==1.13 pymupdf==1.27.2.2 psutil==7.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0

In [2]:
!pip install -q -U accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.8 MB/s eta 0:00:00


In [1]:
import platform
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version (PyTorch): {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")
print(f"Python version: {platform.python_version()}")

CUDA available: True
Current CUDA device: 0
CUDA device name: NVIDIA A100-SXM4-40GB
CUDA version (PyTorch): 12.8
PyTorch version: 2.10.0+cu128
Python version: 3.13.15


## STEP 2 — Load Unlimited-OCR

`trust_remote_code=True` is required by this model. Review the downloaded modeling files if you are in a locked-down environment.


In [2]:
import os
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "baidu/Unlimited-OCR"

ocr_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
ocr_model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
ocr_model = ocr_model.eval().cuda()
print("Unlimited-OCR loaded.")

conversation.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


deepencoder.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- deepencoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_deepseekv2.py: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- configuration_deepseek_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- modeling_deepseekv2.py
- configuration_deepseek_v2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/baidu/Unlimited-OCR:
- conversation.py
- deepencoder.py
- modeling_deepseekv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Unlimited-OCR loaded.


## STEP 3 — PDF → images → OCR → cleaned text

Set `PDF_PATH` to the invoice you are processing. Do not rely on a sample filename.


In [3]:
import tempfile
import fitz  # PyMuPDF

def pdf_to_images(pdf_path, dpi=300):
    doc = fitz.open(pdf_path)
    tmp_dir = tempfile.mkdtemp(prefix="pdf_ocr_")
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []
    for i, page in enumerate(doc):
        out = os.path.join(tmp_dir, f"page_{i+1:04d}.png")
        page.get_pixmap(matrix=mat).save(out)
        paths.append(out)
    doc.close()
    return paths

print("pdf_to_images ready.")

pdf_to_images ready.


In [4]:
import sys
from io import StringIO
import os

PDF_PATH = os.environ.get("PDF_PATH", "/content/INEX21000187.pdf")
OCR_OUTPUT_DIR = "/content/output/pdf"
os.makedirs(OCR_OUTPUT_DIR, exist_ok=True)

if not os.path.exists(PDF_PATH):
    print(f"WARNING: {PDF_PATH} not found. Upload the invoice and re-run this cell.")
    image_paths = []
else:
    image_paths = pdf_to_images(PDF_PATH, dpi=350)
    print(f"Rendered {len(image_paths)} page image(s).")

full_raw_ocr = ""

if image_paths:
    print(f"Running OCR on {len(image_paths)} pages simultaneously...")
    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout = mystdout = StringIO()
    sys.stderr = StringIO()
    try:
        ocr_model.infer_multi(
            ocr_tokenizer,
            prompt="<image>Document parsing.",
            image_files=image_paths,
            output_path=OCR_OUTPUT_DIR,
            image_size=1024,
            max_length=15000,
            no_repeat_ngram_size=35,
            ngram_window=1024,
            save_results=True,
        )
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr

    captured_output = mystdout.getvalue()

    # The model sometimes creates an empty or useless result.md. We'll use captured_output directly.
    result_md = os.path.join(OCR_OUTPUT_DIR, "result.md")
    if os.path.exists(result_md):
        os.remove(result_md) # Clean up to avoid confusion

    full_raw_ocr = captured_output

detailed_path = os.path.join(OCR_OUTPUT_DIR, "detailed_invoice_output.txt")
with open(detailed_path, "w", encoding="utf-8") as f:
    f.write(full_raw_ocr)
print(f"Raw OCR saved: {detailed_path} ({len(full_raw_ocr)} chars)")

Rendered 2 page image(s).
Running OCR on 2 pages simultaneously...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Raw OCR saved: /content/output/pdf/detailed_invoice_output.txt (11572 chars)


In [ ]:
print(ocr_model)

UnlimitedOCRForCausalLM(
  (model): UnlimitedOCRModel(
    (embed_tokens): Embedding(129280, 1280)
    (layers): ModuleList(
      (0): DeepseekV2DecoderLayer(
        (self_attn): SlidingWindowLlamaAttention(
          (q_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (k_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (v_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (o_proj): Linear(in_features=1280, out_features=1280, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): DeepseekV2MLP(
          (gate_proj): Linear(in_features=1280, out_features=6848, bias=False)
          (up_proj): Linear(in_features=1280, out_features=6848, bias=False)
          (down_proj): Linear(in_features=6848, out_features=1280, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): DeepseekV2RMSNorm()
        (post_attention_layernorm): DeepseekV2RMSNorm()
      )
      

## **Cleanup extracted data**

In [5]:
import re
import os

detailed_output_file = os.path.join(OCR_OUTPUT_DIR, "detailed_invoice_output.txt")
plain_text_output_file = os.path.join(OCR_OUTPUT_DIR, "plain_invoice_text.txt")

with open(detailed_output_file, "r", encoding="utf-8") as f:
    content = f.read()

cleaned_text = re.sub(r"<\|det\|>.*?<\|/det\|>", "", content, flags=re.DOTALL)
cleaned_text = re.sub(r"<[^>]*>", "", cleaned_text, flags=re.DOTALL)
cleaned_text = re.sub(r"\n\s*\n", "\n", cleaned_text).strip()

with open(plain_text_output_file, "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print(f"Cleaned text saved: {plain_text_output_file} ({len(cleaned_text)} chars)")
print("--- preview ---")
print(cleaned_text[:1500] if cleaned_text else "[empty]")

Cleaned text saved: /content/output/pdf/plain_invoice_text.txt (4831 chars)
--- preview ---
S.C.R. Engineers Ltd.
Hamełacha St. 18, Netanya, ISRAEL
P.O.B. 13564, Hadarin, Netanya 4237927
Tel.: 972-9-8652050, Fax: 972-9-8659073
Company Number: 510728967
VAT Number: 557167996
Withheld Tax File: 921066858
Reported for VAT Consol. Busn Number 557167996
[Non-Text]
SCR by Allflex
Make every cow count
URL: http://www.scrdairy.com
E-mail: sales@scrdairy.com
To:
Allflex Australia
33 Neumann Rd
Capalaba Qld 4157
Attn: Julie Cornelius
Tel.: +61 7 3245 9100
Australia
From Warehouse: 24, Israel
Ship to:
Allflex Australia
33 Neumann Rd
Capalaba Qld 4157 Australia
Invoice Date: 02/14/21
Return Date: 02/14/21
Receipt Date: 02/14/21
Print Date: 02/14/21
Print Time: 18:20
Document Number: EX21000187
Export Invoice EX21000187 - Copy
OrderYour OrderPart NumberPart DescriptionQuantityUnitUnit PriceExtended PriceCountry Of OriginNet WeightPRODUCT NAMEC20004016POR7722DFS00031DF Master Card 220V DF80010.00eaU

## STEP 4 — Hugging Face auth + load Qwen2.5-7B-Instruct

Add a Colab secret named `HF_TOKEN` (key icon) **or** export `HF_TOKEN` in the environment. Do not paste tokens into cells.


In [6]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found. Public models may still load; gated models will fail.")

Logged in to Hugging Face.


In [7]:
from transformers import AutoModelForCausalLM

LLM_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {LLM_ID}...")
Qtokenizer = AutoTokenizer.from_pretrained(LLM_ID)
Qmodel = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
Qmodel.eval()
print("Qwen loaded.")

Loading Qwen/Qwen2.5-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen loaded.


In [ ]:
print(Qmodel)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

## STEP 5 — Extract invoice JSON only

Qwen reads this PDF only. It must not classify and must not assume a country that is not printed on the document.


### **Connect to the Database**

## Re-structure the extracted data.

In [8]:
system_prompt = """
You are an expert data extraction and product tariff classification assistant.

Your task is to convert the provided OCR text into the required structured JSON format.

CRITICAL INSTRUCTION FOR MULTI-PAGE INVOICES:
The provided OCR text contains multiple pages of items. You MUST extract EVERY SINGLE line item across ALL pages.
Do not skip, summarize, or truncate the list of items. It is strictly forbidden to leave out any items. Continue extracting until the very last item on the last page is included in the JSON.

IMPORTANT:
You must distinguish between:
1. SKU / Part Number
2. HS Code
3. HTSUS Code
4. Product description

Never treat a SKU, Part Number, model number, UPC, EAN, GTIN, invoice number, or catalog number as an HS/HTS tariff code.

SKU RULE:
When extracting data for `item_level`:
- If an explicit `sku_code` is found, use it.
- If an explicit `sku_code` is not found, use the value from the `Part Number` field as `sku_code`.
- OCR TEXT MERGE ISSUE: Due to OCR formatting, the 'Order', 'Your Order', and 'Part Number' values might be concatenated into a single string without spaces (e.g., 'C20004016POR7722DFS00031'). You must logically split this string and ONLY extract the actual 'Part Number' portion (e.g., 'DFS00031', 'DFS00079', 'IDU00034') to use as the `sku_code`. Do not include the preceding order numbers in the sku_code.
- Do not use an HTS/HS code as the SKU.

HTS/HS CLASSIFICATION RULES:

If an HS/HTS code is explicitly present in the OCR text:
- Extract it only if it is clearly identified as an HS Code, Harmonized Code, HTS Code, HTSUS Code, Tariff Code, Customs Code, or equivalent.
- Do not assume that a numeric value is an HS/HTS code merely because it contains 6, 8, or 10 digits.

If an HS/HTS code is NOT explicitly present:
- Do NOT invent a tariff code.
- Do NOT use a SKU or Part Number as an HS/HTS code.
- Do NOT select a code simply because it appears in a Google search result.
- Do NOT assume that the first search result is the correct classification.
- If external web search/retrieval is available, use authoritative tariff sources and customs rulings to determine the classification.
- Prefer official government tariff databases and customs classification rulings over commercial websites.
- If the available product information is insufficient to determine the tariff classification, return null/unknown according to the schema rather than hallucinating a code.
- NOTE: `manufacturing_country_hts_code` means the HTS/HS code of the product in the origin/manufacturing country. `destination_country_hts_code` is the HTS/HS code for the destination country. They are actual numeric HTS/HS codes, NOT country codes (e.g., do NOT output 'IL' or 'AU'). If the specific HTS/HS code is not explicitly in the invoice, return null.

PRODUCT CLASSIFICATION:
Before determining an HS/HTS code, identify:
- What the product physically is
- Primary function
- Intended use
- Whether it is a standalone product or part/accessory of another product
- Material/composition
- Construction
- Whether it is sold separately
- Country/import jurisdiction, if available

Do not classify a product solely from its product name.

VALIDATION:
Before returning an HS/HTS code, verify:
1. The code exists in the applicable tariff nomenclature.
2. The code's official description is compatible with the product.
3. The product's material and function are compatible with the heading.
4. The code is appropriate for the specified import country.
5. The code is not merely a SKU, part number, model number, or unrelated numeric identifier.

For US imports:
- Use the US HTSUS classification.
- Distinguish the international 6-digit HS code from the US 8/10-digit HTSUS code.
- Do not fabricate the final 10-digit HTSUS suffix.
- If only the 6-digit HS classification can be established reliably, return the 6-digit code and indicate that the complete HTSUS classification requires further verification.

CONFIDENCE:
Return a confidence value for tariff classification:
- HIGH: Explicit tariff code found in authoritative source or classification is strongly supported by an official ruling.
- MEDIUM: Classification is strongly supported by product characteristics but the exact tariff provision requires verification.
- LOW: Insufficient information or competing classifications exist.

EVIDENCE:
When classification is inferred rather than explicitly provided, include:
- classification_reason
- source/evidence used, if available
- confidence

HALLUCINATION PREVENTION:
Never create a tariff code merely to fill a field.
If the correct value cannot be determined from the available information, return null.

OUTPUT:
Return ONLY valid JSON matching the schema below.
Do not add explanations, markdown, or text outside the JSON.

Use this exact schema:

{
    "invoices": [
        {
            "order_level": {
                "company_name": string or null,
                "company_address": string or null,
                "invoice_number": string or null,
                "invoice_date": string or null,
                "purchase_order_number": string or null,
                "currency": string or null,
                "payment_terms": string or null,
                "inco_terms": string or null,
                "tax_id": string or null,
                "tracking_number": string or null,
                "shipper_company": string or null, // This is the company originating the shipment (e.g., the invoice issuer)
                "receiver_company": string or null,
                "receiver_firstname": string or null,
                "receiver_lastname": string or null,
                "receiver_phone_number": string or null,
                "receiver_country": string or null, // This is the country of the shipment's final destination
                "receiver_city": string or null, // This is the city of the shipment's final destination
                "receiver_zipcode": string or null, // This is the zipcode of the shipment's final destination
                "receiver_address1": string or null, // This is the primary address line of the shipment's final destination
                "total_amount": float or 0.0
            },
            "item_level": [
                {
                    "sku_code": string or null,
                    "manufacturing_country_hts_code": string or null, // The HTS/HS code of the product in the origin/manufacturing country. Numeric code, NOT a country code. Return null if not explicitly in the invoice.
                    "destination_country_hts_code": string or null, // The HTS/HS code of the product in the destination country. Numeric code, NOT a country code. Return null if not explicitly in the invoice.
                    "hs_code": string or null,
                    "classification_reason": string or null,
                    "product_description": string or null,
                    "simplified_description": string or null,
                    "country_of_manufacture": string or null, // Extract from product description if available (e.g., 'CHINE', 'FRANCE'). If not explicitly mentioned for the item, use the country from the 'company_address' (if country of origin is mentioned seperatly) in 'order_level'.
                    "gross_value": float or null,
                    "number_of_units_for_this_sku": int or null,
                    "unit_value_for_this_sku": float or null,
                    "extended_value_for_this_sku": float or null
                }
            ],
            "shipment": {
                "invoice_number": string or null,
                "invoice_date": string or null,
                "currency": string or null
            }
        }
    ]
}

Return ONLY valid JSON. Do not include markdown formatting like ```json. Do not include explanations.
"""

### Connect to Weaviate Vector Database

In [9]:
!pip install -q weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.5/660.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.84.0 requires grpcio>=1.84.0, but you have grpcio 1.78.0 which is incompatible.


In [10]:
import weaviate
from google.colab import userdata

WEAVIATE_HOST = "32.236.127.40"
WEAVIATE_HTTP_PORT = 8080 # Assuming HTTP port 8080 for Weaviate
WEAVIATE_GRPC_PORT = 50051 # Assuming gRPC port 50051 for Weaviate

# try:
#     # API key for secure access to Weaviate, if required
#     weaviate_api_key = userdata.get("WEAVIATE_API_KEY")
# except userdata.SecretNotFoundError:
#     print("WEAVIATE_API_KEY secret not found. Connecting without API key.")
#     weaviate_api_key = None


try:
    print(f"Connecting to Weaviate at {WEAVIATE_HOST}:{WEAVIATE_HTTP_PORT} (HTTP) and {WEAVIATE_GRPC_PORT} (gRPC)...")
    auth_config = None
    # if weaviate_api_key:
    #     auth_config = weaviate.auth.AuthApiKey(api_key=weaviate_api_key)

    # Attempt to connect using the HTTP port first
    weaviate_client = weaviate.connect_to_custom(
    http_host=WEAVIATE_HOST,
    http_port=8080,
    http_secure=False,
    grpc_host=WEAVIATE_HOST,
    grpc_port=50051,
    grpc_secure=False,
)

    if weaviate_client.is_live():
        print("Successfully connected to Weaviate (HTTP)!")
        print(f"Weaviate cluster status: {weaviate_client.get_meta()['version']}") # Access version as a dictionary key

        # Use client.collections.list_all() for the new client
        collections = weaviate_client.collections.list_all()
        collection_names = [c.name for c in collections.values()]
        print(f"Available collections (Weaviate): {collection_names}")
    else:
        print("Failed to connect to Weaviate (HTTP).")
        weaviate_client = None

except Exception as e:
    print(f"Failed to connect to Weaviate: {e}")
    weaviate_client = None

Connecting to Weaviate at 32.236.127.40:8080 (HTTP) and 50051 (gRPC)...
Successfully connected to Weaviate (HTTP)!
Weaviate cluster status: 1.38.13
Available collections (Weaviate): ['Hs_category']


In [11]:
import json

def qwen_generate(messages, max_new_tokens=8000, temperature=0.0):
    prompt_text = Qtokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = Qtokenizer([prompt_text], return_tensors="pt").to(Qmodel.device)
    with torch.no_grad():
        out = Qmodel.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=Qtokenizer.eos_token_id,
            pad_token_id=Qtokenizer.eos_token_id,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
        )
    return Qtokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True).strip()


def parse_json_maybe(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.IGNORECASE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise
        return json.loads(m.group(0))


text_to_format = cleaned_text if "cleaned_text" in globals() and cleaned_text else ""
if not text_to_format:
    raise RuntimeError("cleaned_text is empty. Run the OCR cells first.")

print("Generating invoice JSON (no tariff classification)...")
raw_json = qwen_generate(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"OCR Text:\n\n{text_to_format}"},
    ],
    max_new_tokens=40000,
    temperature=0.0,
)
parsed_json = parse_json_maybe(raw_json)

# Hard-clear any leaked tariff guesses from extraction
for inv in parsed_json.get("invoices", []):
    for item in inv.get("item_level", []):
        for field in (
            "hs_code",
            "manufacturing_country_hts_code",
            "destination_country_hts_code",
            "classification_reason",
        ):
            val = item.get(field)
            if isinstance(val, str) and re.match(r"^[A-Za-z][A-Za-z\s]*$", val.strip()):
                item[field] = None
        # extraction step must not classify
        item["hs_code"] = None
        item["manufacturing_country_hts_code"] = None
        item["destination_country_hts_code"] = None
        item["classification_reason"] = None

os.makedirs("/content/output", exist_ok=True)
extract_path = "/content/output/extracted_invoice.json"
with open(extract_path, "w", encoding="utf-8") as f:
    json.dump(parsed_json, f, indent=2)
print(f"Saved {extract_path}")
print(json.dumps(parsed_json, indent=2)[:3000])

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating invoice JSON (no tariff classification)...
Saved /content/output/extracted_invoice.json
{
  "invoices": [
    {
      "order_level": {
        "company_name": "S.C.R. Engineers Ltd.",
        "company_address": "Hame\u0142acha St. 18, Netanya, ISRAEL\nP.O.B. 13564, Hadarin, Netanya 4237927",
        "invoice_number": "EX21000187",
        "invoice_date": "02/14/21",
        "purchase_order_number": "Not Provided",
        "currency": "USD",
        "payment_terms": "Not Provided",
        "inco_terms": "Not Provided",
        "tax_id": "557167996",
        "tracking_number": "Not Provided",
        "shipper_company": "S.C.R. Engineers Ltd.",
        "receiver_company": "Allflex Australia",
        "receiver_firstname": "Julie",
        "receiver_lastname": "Cornelius",
        "receiver_phone_number": "+61 7 3245 9100",
        "receiver_country": "Australia",
        "receiver_city": "Capalaba",
        "receiver_zipcode": "4157",
        "receiver_address1": "33 Neumann Rd

### DB connection

In [ ]:
!pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 20.5 MB/s eta 0:00:00


In [ ]:
import psycopg2
import pandas as pd
from google.colab import userdata

try:
    # Ensure these are set in your Colab Secrets (the 🔑 icon on the left)
    db_host = userdata.get("DB_HOST")
    db_name = userdata.get("DB_NAME")
    db_user = userdata.get("DB_USER")
    db_password = userdata.get("DB_PASSWORD")
    db_port = userdata.get("DB_PORT")
except userdata.SecretNotFoundError as e:
    print(f"Secret not found: {e}. Please add it to your Colab secrets.")
    # Fallback defaults to prevent immediate crash if testing locally without secrets
    db_host, db_name, db_user, db_password, db_port = "localhost", "postgres", "postgres", "", "5432"

try:
    print("Connecting to PostgreSQL...")
    # conn = psycopg2.connect(
    #     host=db_host,
    #     database=db_name,
    #     user=db_user,
    #     password=db_password,
    #     port=db_port
    # )
    # print("Successfully connected to the PostgreSQL database!")
    print("PostgreSQL connection commented out as Weaviate is now used for HTS lookup.")

    # Example of querying the database for HTS codes:
    # query = "SELECT * FROM hts_codes LIMIT 5;"
    # df_hts = pd.read_sql(query, conn)
    # display(df_hts)

except Exception as e:
    print(f"Failed to connect to the database: {e}")

Connecting to PostgreSQL...
PostgreSQL connection commented out as Weaviate is now used for HTS lookup.


### Connect to PostgreSQL and Query HTS Code

In [ ]:
import psycopg2
import pandas as pd
from google.colab import userdata

pg_conn = None # Initialize pg_conn to None globally to ensure it's always defined

try:
    # Ensure these are set in your Colab Secrets (the 🔑 icon on the left)
    db_host = userdata.get("DB_HOST")
    db_name = userdata.get("DB_NAME")
    db_user = userdata.get("DB_USER")
    db_password = userdata.get("DB_PASSWORD")
    db_port = userdata.get("DB_PORT")
except userdata.SecretNotFoundError as e:
    print(f"Secret not found: {e}. Please add it to your Colab secrets.")
    db_host, db_name, db_user, db_password, db_port = "localhost", "postgres", "postgres", "", "5432"

try:
    print("Connecting to PostgreSQL...")
    pg_conn = psycopg2.connect(
        host=db_host,
        database=db_name,
        user=db_user,
        password=db_password,
        port=db_port
    )
    print("Successfully connected to the PostgreSQL database!")
except Exception as e:
    print(f"Failed to connect to the PostgreSQL database: {e}")
    pg_conn = None


Connecting to PostgreSQL...
Successfully connected to the PostgreSQL database!


In [ ]:
if pg_conn:
    search_hts_code = "8517622"
    # Assuming your HTS codes are in a table named 'hts_codes' with a column 'code'
    # You might need to adjust the table and column names based on your database schema.
    query = f"SELECT * FROM Hs_category WHERE code = '{search_hts_code}';"

    try:
        df_hts_pg = pd.read_sql(query, pg_conn)
        if not df_hts_pg.empty:
            print(f"HTS code '{search_hts_code}' found in PostgreSQL:")
            display(df_hts_pg)
        else:
            print(f"HTS code '{search_hts_code}' not found in PostgreSQL.")
    except Exception as e:
        print(f"Error querying PostgreSQL for HTS code '{search_hts_code}': {e}")
    finally:
        # pg_conn.close()
        print("PostgreSQL connection closed.")
else:
    print("Cannot query PostgreSQL: No active connection.")

/tmp/ipykernel_18380/3759574047.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hts_pg = pd.read_sql(query, pg_conn)


HTS code '8517622' found in PostgreSQL:


,id,country_code,description,code,indent,reference,is_leaf,isactive,isdelete,createdon
0,163130,31,Apparatus for conmutation of telephonic lines,8517622,4,851762,False,True,False,2025-07-28 11:24:27.559811+00:00


PostgreSQL connection closed.


In [ ]:
display(df_hts_pg)

,id,country_code,description,code,indent,reference,is_leaf,isactive,isdelete,createdon
0,163130,31,Apparatus for conmutation of telephonic lines,8517622,4,851762,False,True,False,2025-07-28 11:24:27.559811+00:00


In [ ]:
pg_conn.close()

### Map Country Codes to Country Names
Paste your JSON payload into the string below to map the numeric `country_code` to a human-readable `country_name`.

### Dynamic Extraction & API Mapping
Fetch countries from the Logistx API, extract the country and product name from the JSON, and query the database.

In [12]:
import requests
import pandas as pd
import re
import json
import os
from IPython.display import display, JSON

# Ensure weaviate_client is available from previous cells
if 'weaviate_client' not in globals() or not weaviate_client:
    raise RuntimeError("Weaviate client not connected. Please run the Weaviate connection cells first.")

# Add a helper function to fetch children
def get_children_from_weaviate(parent_hs_code, country_code, hs_collection):
    children_rows = []
    try:
        # 1. Get the indent of the parent HTS code
        parent_obj_response = hs_collection.query.fetch_objects(
            filters=weaviate.classes.query.Filter.by_property("code").equal(parent_hs_code) &
                    weaviate.classes.query.Filter.by_property("country_code").equal(country_code),
            return_properties=["indent"],
            limit=1
        )
        if not parent_obj_response.objects:
            # Parent not found or no indent for parent, cannot determine children by indent
            print(f"    -> Warning: Parent HS code {parent_hs_code} not found or no indent for this country {country_code}.")
            return []

        parent_indent = int(parent_obj_response.objects[0].properties.get("indent", 0)) # Default to 0 if indent is missing

        # 2. Query for potential children based on code prefix, country_code, and indent level
        # HTS codes are hierarchical, so children codes usually start with the parent's code.
        response = hs_collection.query.fetch_objects(
            filters=weaviate.classes.query.Filter.by_property("code").like(f"{parent_hs_code}*") &
                    weaviate.classes.query.Filter.by_property("country_code").equal(country_code) &
                    weaviate.classes.query.Filter.by_property("indent").equal(str(parent_indent + 1)), # Filter for direct children indent
            return_properties=["country_code", "code", "description", "indent"],
        )
        for obj in response.objects:
            props = obj.properties
            child_code = props.get("code")
            # Ensure it's a child (longer code) and not the parent itself
            if child_code and len(child_code) > len(parent_hs_code):
                children_rows.append({
                    "id": str(obj.uuid),
                    "country_code": props.get("country_code"),
                    "code": child_code,
                    "description": props.get("description"),
                    "indent": props.get("indent"),
                    "rank": -1, # Assign a very low rank for inferred children
                    "is_inferred_child": True # Flag to indicate it's an inferred child
                })
    except Exception as e:
        print(f"    -> Error fetching children for parent {parent_hs_code}: {e}")
    return children_rows


# 1. Fetch countries from the Logistx API
api_url = "https://bapi.logistx.us/his_countries"
try:
    response = requests.get(api_url)
    response.raise_for_status()
    countries_data = response.json()
    countries_list = countries_data.get("countries", [])
    country_map = {}
    for c in countries_list:
        name = str(c.get('country_name', '')).strip().upper()
        code = str(c.get('country_code', '')).strip()
        if name and code:
            country_map[name] = code
            base_name = name.split("-")[0].strip()
            if base_name: country_map[base_name] = code
    print(f"Successfully loaded {len(country_map)} countries from the API.")
except Exception as e:
    print(f"Error fetching countries from API: {e}")
    country_map = {}

# 2. Extract country and product descriptions from JSON, then query Weaviate
if 'parsed_json' in globals() and 'invoices' in parsed_json:
    invoice = parsed_json['invoices'][0]
    order_level = invoice.get('order_level', {})
    receiver_country = str(order_level.get('receiver_country', '')).strip().upper()
    dest_country_code = country_map.get(receiver_country, receiver_country)
    shipper_country = str(order_level.get('shipper_country', '')).strip().upper()

    items = invoice.get('item_level', [])
    db_matches_output = []

    STOP_WORDS = {'THE', 'AND', 'FOR', 'WITH', 'USED', 'THEIR', 'FROM', 'THIS', 'THAT', 'ITS', 'INDUSTRY'}

    # Get the Weaviate collection for Hs_category
    hs_category_collection = weaviate_client.collections.get("Hs_category")

    for i, item in enumerate(items, 1):
        desc = item.get('product_description', '')
        if not desc: continue

        item_origin = str(item.get('country_of_manufacture', '')).strip().upper()
        origin_country = item_origin if item_origin else shipper_country
        orig_country_code = country_map.get(origin_country, origin_country) if origin_country else ""

        item_matches = {"sku_code": item.get("sku_code"), "product_description": desc, "matches": {}}

        words = [w.upper() for w in re.findall(r'[a-zA-Z]+', desc) if len(w) > 2]
        filtered_words = [w for w in words if w not in STOP_WORDS]

        if not filtered_words:
            db_matches_output.append(item_matches)
            continue

        search_query_text = ' '.join(filtered_words) # Combined words for Weaviate search

        codes_to_search = []
        if dest_country_code: codes_to_search.append(('Destination', dest_country_code))
        if orig_country_code and orig_country_code != dest_country_code:
            codes_to_search.append(('Origin', orig_country_code))

        for c_type, c_code in codes_to_search:
            try:
                # Weaviate query using bm25 (keyword search)
                response = hs_category_collection.query.bm25(
                    query=search_query_text,
                    filters=weaviate.classes.query.Filter.by_property("country_code").equal(c_code),
                    return_properties=["country_code", "code", "description", "indent"],
                    return_metadata=weaviate.classes.query.MetadataQuery(score=True)
                )

                rows_weaviate = []
                seen_codes = set() # To prevent adding duplicate codes

                # First, add direct bm25 matches
                for obj in response.objects:
                    props = obj.properties
                    code = props.get("code")
                    if code and code not in seen_codes:
                        rank = obj.metadata.score if obj.metadata.score is not None else 0.0
                        rows_weaviate.append({
                            "id": str(obj.uuid),
                            "country_code": props.get("country_code"),
                            "code": code,
                            "description": props.get("description"),
                            "indent": props.get("indent"),
                            "rank": rank,
                            "is_inferred_child": False
                        })
                        seen_codes.add(code)

                # Then, for each direct match, fetch its children
                # Create a copy of codes from direct matches to iterate, as we are adding to rows_weaviate
                direct_matches_codes = [r["code"] for r in rows_weaviate if not r["is_inferred_child"]] # Only process direct matches for children
                for parent_code in direct_matches_codes:
                    children = get_children_from_weaviate(parent_code, c_code, hs_category_collection)
                    for child in children:
                        code = child["code"]
                        if code and code not in seen_codes:
                            rows_weaviate.append(child)
                            seen_codes.add(code)

                # Sort by rank descending. Inferred children with rank -1 will appear at the bottom.
                # Then by id ascending for tie-breaking.
                rows_weaviate.sort(key=lambda x: (x['rank'], x['id']), reverse=True)

                if rows_weaviate:
                    data = rows_weaviate
                    tree, stack = [], {}
                    for match in data:
                        lvl = int(match.get('indent', 1) or 1)
                        node = {**match, 'children': []}
                        if lvl <= 1:
                            tree.append(node)
                        else:
                            p_lvl = lvl - 1
                            while p_lvl > 0 and p_lvl not in stack: p_lvl -= 1
                            if p_lvl > 0: stack[p_lvl]['children'].append(node)
                            else: tree.append(node)
                        stack[lvl] = node
                    item_matches['matches'][c_type] = tree
                else:
                    item_matches['matches'][c_type] = []
            except Exception as e:
                print(f"    -> Weaviate Query failed: {e}")

        db_matches_output.append(item_matches)

    display(JSON(db_matches_output))
    os.makedirs('/content/output', exist_ok=True)
    with open('/content/output/db_matches.json', 'w', encoding='utf-8') as f:
        json.dump(db_matches_output, f, indent=2, default=str)
else:
    print("parsed_json is missing.")

Successfully loaded 421 countries from the API.


<IPython.core.display.JSON object>

In [13]:
import json
import re
from IPython.display import display, JSON

if 'Qmodel' not in globals() or 'Qtokenizer' not in globals():
    print("Warning: Qwen model is not in memory. Please re-run the STEP 4 cell to load it.")
else:
    print("Using Qwen to analyze DB matches and assign HTS codes...\n")

    db_matches_map = {}
    if 'db_matches_output' in globals():
        for match in db_matches_output:
            sku = match.get('sku_code')
            if sku:
                db_matches_map[sku] = match.get('matches', {})
    else:
        print("Warning: db_matches_output not found.")

    invoice = parsed_json['invoices'][0]
    items = invoice.get('item_level', [])

    for i, item in enumerate(items, 1):
        sku = item.get('sku_code')
        desc = item.get('product_description')
        matches = db_matches_map.get(sku, {})

        dest_matches = matches.get('Destination', [])
        orig_matches = matches.get('Origin', [])

        # Helper to flatten the tree for the prompt
        def flatten_tree(nodes, depth=0):
            flat = []
            for n in nodes:
                flat.append(f"- {n['code']}: {n['description']} (Rank: {n.get('rank', 0)})")
                if 'children' in n and n['children']:
                    flat.extend(flatten_tree(n['children'], depth + 1))
            return flat

        dest_cands = "\n".join(flatten_tree(dest_matches)[:15]) if dest_matches else "None"
        orig_cands = "\n".join(flatten_tree(orig_matches)[:15]) if orig_matches else "None"

        prompt = f"""Analyze the product and select the best HTS code from the candidates.
Product: {desc}
SKU: {sku}

Origin Candidates:
{orig_cands}

Destination Candidates:
{dest_cands}

Return JSON ONLY:
{{
  "manufacturing_country_hts_code": "code or null",
  "destination_country_hts_code": "code or null",
  "classification_reason": "explanation"
}}
"""

        try:
            raw_response = qwen_generate([
                {"role": "system", "content": "You are a customs assistant. Output JSON ONLY."},
                {"role": "user", "content": prompt}
            ], max_new_tokens=500)

            result = parse_json_maybe(raw_response)
            item.update(result)
            print(f"[{i}/{len(items)}] SKU: {sku} -> Dest: {item['destination_country_hts_code']}")
        except Exception as e:
            print(f"Error on SKU {sku}: {e}")

    display(JSON(parsed_json))
    with open('/content/output/extracted_invoice_with_hts.json', 'w') as f:
        json.dump(parsed_json, f, indent=2)

Using Qwen to analyze DB matches and assign HTS codes...

[1/38] SKU: DFS00031 -> Dest: 8448310027
[2/38] SKU: DFS00079 -> Dest: 8481809037
[3/38] SKU: IDU00034 -> Dest: 8459100021
[4/38] SKU: CBX00030 -> Dest: 8434100012
[5/38] SKU: CBX00039 -> Dest: 3923109050
[6/38] SKU: FFM00053 -> Dest: 0104100010
[7/38] SKU: FFS00013 -> Dest: 8426190011
[8/38] SKU: FFS00027 -> Dest: 8433591039
[9/38] SKU: MPP00017 -> Dest: 85442000
[10/38] SKU: MPP00022 -> Dest: 8528590067
[11/38] SKU: MPP00036 -> Dest: 8459100021
[12/38] SKU: MPS00013 -> Dest: 9102110042
[13/38] SKU: PUL00233 -> Dest: 3215110008
[14/38] SKU: PUL00237 -> Dest: 3215110008
[15/38] SKU: PUL00234 -> Dest: 3215110008
[16/38] SKU: PUL00239 -> Dest: 3215110008
Error on SKU PUL00240: Unterminated string starting at: line 4 column 28 (char 121)
[18/38] SKU: PHOTO CELL PNP -> Dest: 8540200005
[19/38] SKU: PUL90028 -> Dest: None
[20/38] SKU: PUL90069 -> Dest: None
[21/38] SKU: PUL90071 -> Dest: None
[22/38] SKU: PUL90079 -> Dest: 85176
[23/

<IPython.core.display.JSON object>

In [14]:
if 'weaviate_client' not in globals() or not weaviate_client:
    print("Weaviate client not connected. Please run the Weaviate connection cells first.")
else:
    hs_category_collection = weaviate_client.collections.get("Hs_category")
    search_code = "8537"
    # search_code = "8517622000"

    try:
        # Query by property 'code'
        response = hs_category_collection.query.fetch_objects(
            filters=weaviate.classes.query.Filter.by_property("code").equal(search_code),
            return_properties=["country_code", "code", "description", "indent"]
        )

        if response.objects:
            print(f"HTS code '{search_code}' found in the database:")
            for obj in response.objects:
                props = obj.properties
                print(f"  Country Code: {props.get('country_code')}, Code: {props.get('code')}, Description: {props.get('description')}, Indent: {props.get('indent')}")
        else:
            print(f"HTS code '{search_code}' not found in the database.")

    except Exception as e:
        print(f"Error querying Weaviate for HTS code '{search_code}': {e}")

HTS code '8537' found in the database:
  Country Code: 14, Code: 8537, Description: boards, panels, consoles, desks, cabinets and other bases, equipped with two or more apparatus of 8535 or 8536, for electric control or the distribution of electricity, including those incorporating instruments or apparatus of chapter 90, and numerical control apparatus, other than switching apparatus of 8517, Indent: 2
  Country Code: 1, Code: 8537, Description: boards, panels, consoles, desks, cabinets and other bases, equipped with two or more apparatus of heading 8535 or 8536, for electric control or the distribution of electricity, including those incorporating instruments or apparatus of chapter 90, and numerical control apparatus, other than switching apparatus of heading 8517, Indent: 2
  Country Code: 14, Code: 8537, Description: Boards, panels, consoles, desks, cabinets and other bases, equipped with two or more apparatus of 8535 or 8536, for electric control or the distribution of electricity

In [ ]:
from IPython.display import display, HTML

# Prepare data for display
display_data = []
for invoice in parsed_json.get('invoices', []):
    for item in invoice.get('item_level', []):
        display_data.append({
            'SKU Code': item.get('sku_code'),
            'Product Description': item.get('product_description'),
            'Manufacturing Country HTS Code': item.get('manufacturing_country_hts_code'),
            'Destination Country HTS Code': item.get('destination_country_hts_code')
        })

# Convert to DataFrame for better tabular display
df_summary = pd.DataFrame(display_data)

print("Summary of HTS Codes in Invoice JSON:")
display(df_summary)

Summary of HTS Codes in Invoice JSON:


,SKU Code,Product Description,Manufacturing Country HTS Code,Destination Country HTS Code
0,DFS00031,DF Master Card 220V,null,8448310027
1,DFS00079,DF Sorting Gate Control Box IR,null,8474100022
2,IDU00034,LD ID Unit IDU510,null,8459100021
3,CBX00030,MC200 Milking/Rinsing Control,null,8434100012
4,CBX00039,MC200/ED Connection Box CB200A,null,3923109050
5,FFM00053,Fat Sampler FFS Cow/sheep,None,0104100010
6,FFS00013,MC200 FF Sensor FFS30C 24V Green 3M Cable,null,8544200017
7,FFS00027,MC200 FF Sensor FFS30 24V Green,None,0711200020
8,MPP00017,Pushbutton PPB30 N.C Blue Cable 3.5M,None,3204170018
9,MPP00022,Two Way Milk Valve MV30 Black,None,3502200021


In [ ]:
if 'db_matches_output' in globals():
    print(f"--- Updated Search Results Summary ---")
    for item in db_matches_output:
        sku = item.get('sku_code')
        desc = item.get('product_description')
        matches = item.get('matches', {})
        dest_count = len(matches.get('Destination', []))
        orig_count = len(matches.get('Origin', []))
        print(f"SKU: {sku}")
        print(f"Description: {desc}")
        print(f"Destination Country Matches: {dest_count}")
        print(f"Origin Country Matches: {orig_count}\n")

        if dest_count > 0:
             print(f"Top Destination Matches (Ranked):")
             # Only show top 5 to keep the chat readable
             for m in matches.get('Destination', [])[:5]:
                 print(f"  - {m['code']}: {m['description']} (Rank: {m.get('rank', 'N/A')})")
             print("")
else:
    print("db_matches_output not found. Please run the database search cell first.")

--- Updated Search Results Summary ---
SKU: 700921
Description: UV MWP PROCESS MAGENTA SLF15 Ink used in the printing industry
Destination Country Matches: 15
Origin Country Matches: 14

Top Destination Matches (Ranked):
  - 3215: Printing ink, writing or drawing ink and other inks, whether or not concentrated or solid (Rank: 0.023923056)
  - 9612: Typewriter or similar ribbons, inked or otherwise prepared for giving impressions, whether or not on spools or in cartridges; ink-pads, whether or not inked, with or without boxes (Rank: 0.013790939)
  - 49: Printed books, newspapers, pictures and other products of the printing industry; manuscripts, typescripts and plans (Rank: 0.012665148)
  - 4901: Printed books, brochures, leaflets and similar printed matter, whether or not in single sheets (Rank: 0.012665148)
  - 4909: Printed or illustrated postcards; printed cards bearing personal greetings, messages or announcements, whether or not illustrated, with or without envelopes or trimmings 

In [ ]:
import json

if 'countries_data' in globals():
    # Print the structure of the API response (limiting output length to avoid crashing the output)
    print("API Response Data:\n")
    print(json.dumps(countries_data, indent=2)[:3000])

    if isinstance(countries_data, dict):
        print(f"\nKeys in response: {list(countries_data.keys())}")
else:
    print("countries_data is not in memory. Please re-run the API fetch cell.")


API Response Data:

{
  "statuscode": 1,
  "countries": [
    {
      "id": 1,
      "country_code": "2",
      "country_name": "AFGHANISTAN - AF",
      "isactive": true,
      "createdon": "2025-06-27T07:17:49.890051+00:00"
    },
    {
      "id": 2,
      "country_code": "3",
      "country_name": "ALBANIA - AL",
      "isactive": true,
      "createdon": "2025-06-27T07:17:49.890051+00:00"
    },
    {
      "id": 3,
      "country_code": "4",
      "country_name": "ALGERIA - DZ",
      "isactive": true,
      "createdon": "2025-06-27T07:17:49.890051+00:00"
    },
    {
      "id": 4,
      "country_code": "6",
      "country_name": "ANDORRA - AD",
      "isactive": true,
      "createdon": "2025-06-27T07:17:49.890051+00:00"
    },
    {
      "id": 5,
      "country_code": "7",
      "country_name": "ANGOLA - AO",
      "isactive": true,
      "createdon": "2025-06-27T07:17:49.890051+00:00"
    },
    {
      "id": 6,
      "country_code": "8",
      "country_name": "ANGUILLA - AI

## STEP 6 — Load a generic international HS catalog (not a product list)

Downloads the public UN Comtrade / datapackage HS nomenclature (section, chapter, heading, subheading titles).

This is **not** product hardcoding. It is the shared 2/4/6-digit language every customs administration uses.

Optional overlays, if present:
- `/content/tariff_index_national.json` — official national lines you maintain
  (`jurisdiction`, `hs6`, `national_code`, `title`, `legal_text`)
- Destination country on the invoice selects which overlay to use.
- If no overlay exists for that country, return HS-6 only.


In [ ]:
import csv
import io
import urllib.request
from collections import defaultdict

HS_CSV_URLS = [
    "https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv",
    "https://cdn.jsdelivr.net/gh/datasets/harmonized-system@main/data/harmonized-system.csv",
]

COUNTRY_ALIASES = {
    # Names and common aliases -> ISO2. Used only to pick a schedule file, never to invent a code.
    "united states": "US", "usa": "US", "u.s.": "US", "u.s.a.": "US", "america": "US",
    "united states of america": "US",
    "australia": "AU", "au": "AU",
    "new zealand": "NZ", "nz": "NZ",
    "united kingdom": "GB", "uk": "GB", "great britain": "GB", "england": "GB",
    "canada": "CA", "ca": "CA",
    "germany": "DE", "de": "DE",
    "france": "FR", "fr": "FR",
    "italy": "IT", "it": "IT",
    "spain": "ES", "es": "ES",
    "netherlands": "NL", "holland": "NL",
    "belgium": "BE",
    "ireland": "IE",
    "china": "CN", "prc": "CN", "people's republic of china": "CN",
    "hong kong": "HK",
    "taiwan": "TW",
    "japan": "JP", "jp": "JP",
    "south korea": "KR", "korea": "KR", "republic of korea": "KR",
    "india": "IN", "in": "IN",
    "singapore": "SG",
    "malaysia": "MY",
    "indonesia": "ID",
    "thailand": "TH",
    "vietnam": "VN", "viet nam": "VN",
    "philippines": "PH",
    "mexico": "MX",
    "brazil": "BR",
    "israel": "IL", "il": "IL",
    "united arab emirates": "AE", "uae": "AE",
    "south africa": "ZA",
    "switzerland": "CH",
    "sweden": "SE",
    "norway": "NO",
    "denmark": "DK",
    "poland": "PL",
    "turkey": "TR", "turkiye": "TR",
}


def iso2_from_country(value):
    if not value or str(value).strip().lower() in {"null", "none", "unknown"}:
        return None
    raw = str(value).strip()
    if len(raw) == 2 and raw.isalpha():
        return raw.upper()
    key = raw.lower()
    if key in COUNTRY_ALIASES:
        return COUNTRY_ALIASES[key]
    # last comma-separated token often is the country
    tail = key.split(",")[-1].strip()
    if tail in COUNTRY_ALIASES:
        return COUNTRY_ALIASES[tail]
    for name, iso in COUNTRY_ALIASES.items():
        if name in key:
            return iso
    return None


def load_hs_catalog():
    last_err = None
    text = None
    for url in HS_CSV_URLS:
        try:
            with urllib.request.urlopen(url, timeout=60) as resp:
                text = resp.read().decode("utf-8", errors="replace")
            print(f"Downloaded HS catalog: {url}")
            break
        except Exception as e:
            last_err = e
    local = "/content/harmonized-system.csv"
    if text is None and os.path.exists(local):
        with open(local, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"Loaded local HS catalog: {local}")
    if text is None:
        raise RuntimeError(
            f"Could not download HS catalog and {local} is missing. Last error: {last_err}"
        )
    rows = list(csv.DictReader(io.StringIO(text)))
    return rows


hs_rows = load_hs_catalog()
print(f"HS catalog rows: {len(hs_rows)} columns={list(hs_rows[0].keys()) if hs_rows else []}")

# Normalize common schemas: hscode/description/level/parent  OR section,hscode,description,parent,level
def norm_hs_row(r):
    code = (
        r.get("hscode") or r.get("hs_code") or r.get("code") or r.get("id") or ""
    )
    code = re.sub(r"\D", "", str(code))
    desc = r.get("description") or r.get("text") or r.get("name") or ""
    level = r.get("level") or ""
    parent = r.get("parent") or ""
    parent = re.sub(r"\D", "", str(parent))
    if not level and code:
        level = {2: "chapter", 4: "heading", 6: "subheading"}.get(len(code), "other")
    return {
        "hs_code": code,
        "description": str(desc).strip(),
        "level": str(level).lower(),
        "parent": parent,
        "section": r.get("section") or "",
    }


HS_CATALOG = [norm_hs_row(r) for r in hs_rows]
HS_CATALOG = [r for r in HS_CATALOG if r["hs_code"] and r["description"]]
HS_BY_CODE = {r["hs_code"]: r for r in HS_CATALOG}
HS6 = [r for r in HS_CATALOG if len(r["hs_code"]) == 6]
HS4 = [r for r in HS_CATALOG if len(r["hs_code"]) == 4]
print(f"Usable rows={len(HS_CATALOG)}  HS-4={len(HS4)}  HS-6={len(HS6)}")

# Optional official national overlay (you provide). Schema:
# [{"jurisdiction":"US","hs6":"843490","national_code":"8434.90.00.00","title":"...","legal_text":"..."}]
NATIONAL_INDEX = []
national_path = "/content/tariff_index_national.json"
if os.path.exists(national_path):
    with open(national_path, "r", encoding="utf-8") as f:
        NATIONAL_INDEX = json.load(f)
    print(f"Loaded national overlay: {len(NATIONAL_INDEX)} rows")
else:
    print("No /content/tariff_index_national.json — national 8/10-digit codes will stay null.")

NATIONAL_BY_JUR = defaultdict(list)
for row in NATIONAL_INDEX:
    NATIONAL_BY_JUR[str(row.get("jurisdiction", "")).upper()].append(row)

SCHEDULE_HS6 = {r["hs_code"] for r in HS6}
print("Generic international index ready.")

Downloaded HS catalog: https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv
HS catalog rows: 6940 columns=['section', 'hscode', 'description', 'parent', 'level']
Usable rows=6939  HS-4=1229  HS-6=5613
No /content/tariff_index_national.json — national 8/10-digit codes will stay null.
Generic international index ready.


## STEP 7 — Generic class + retrieve + constrained select

Product class is a **physical/function enum**, not a brand list.  
Destination jurisdiction is read from `receiver_country` on **this** invoice.  
Qwen may only pick an HS-6 that was retrieved from the catalog.


In [ ]:
PRODUCT_CLASSES = (
    "live_animal_or_animal_product",
    "food_or_beverage",
    "mineral_or_chemical",
    "pharmaceutical",
    "plastic_article",
    "rubber_gasket_or_seal",
    "rubber_other",
    "leather_or_hide",
    "wood_or_paper",
    "textile_or_apparel",
    "footwear",
    "stone_ceramic_glass",
    "base_metal_article",
    "complete_machine",
    "machine_part",
    "valve_or_cock",
    "pump_or_compressor",
    "electric_motor_or_generator",
    "printed_circuit",
    "integrated_circuit",
    "control_board_or_panel",
    "switch_or_connector",
    "insulated_cable_or_wire",
    "photocell_or_semiconductor",
    "display_or_indicator",
    "sensor_or_measuring_instrument",
    "lamp_or_lighting",
    "vehicle_or_vehicle_part",
    "optical_or_medical_instrument",
    "furniture",
    "toy_or_sporting",
    "other_manufactured",
    "unknown",
)

# Generic English commodity words only. No vendor, no SKU, no model line.
GENERIC_CLASS_PATTERNS = [
    (r"\bgasket\b|\bwasher\b|\bseal kit\b|\brubber liner\b|\bo-?ring\b", "rubber_gasket_or_seal"),
    (r"\bcable\b|\binsulated wire\b|\bpower cord\b|\bharness\b", "insulated_cable_or_wire"),
    (r"\bprinted circuit\b|\bpcb\b|\bcircuit board\b", "printed_circuit"),
    (r"\bintegrated circuit\b|\bmicrocontroller\b|\bprocessor\b", "integrated_circuit"),
    (r"\bphotocell\b|\bphoto.?cell\b|\bphotodiode\b|\bsolar cell\b", "photocell_or_semiconductor"),
    (r"\bpushbutton\b|\bpush button\b|\bswitch\b|\bconnector\b|\brelay\b", "switch_or_connector"),
    (r"\bcontrol (?:box|panel|board|cabinet)\b|\bconnection box\b|\bplc\b", "control_board_or_panel"),
    (r"\bdisplay\b|\bindicator panel\b|\blcd\b|\bled panel\b", "display_or_indicator"),
    (r"\bvalve\b|\bcock\b|\btap\b", "valve_or_cock"),
    (r"\bpump\b|\bcompressor\b", "pump_or_compressor"),
    (r"\bmotor\b|\bgenerator\b|\balternator\b", "electric_motor_or_generator"),
    (r"\bsensor\b|\btransducer\b|\bmeter\b|\bgauge\b", "sensor_or_measuring_instrument"),
    (r"\bpulsator\b|\bteat cup\b|\bmilking\b", "machine_part"),
    (r"\bpart[s]?\b|\bsnare part\b|\bspare\b|\bcomponent\b", "machine_part"),
    (r"\bapparel\b|\bgarment\b|\btshirt\b|\bshirt\b|\bfabric\b|\btextile\b", "textile_or_apparel"),
    (r"\bshoe\b|\bboot\b|\bfootwear\b", "footwear"),
    (r"\bpharma\b|\btablet\b|\bmedicine\b|\bvaccine\b", "pharmaceutical"),
    (r"\bvehicle\b|\bautomotive\b|\bbrake pad\b|\btyre\b|\btire\b", "vehicle_or_vehicle_part"),
]


def guess_product_class(description, sku=None):
    # SKU is ignored for class assignment on purpose.
    text = (description or "").lower()
    for pat, cls in GENERIC_CLASS_PATTERNS:
        if re.search(pat, text, flags=re.I):
            return cls
    return "unknown"


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+", (text or "").lower()) if len(t) > 1]


def retrieve_hs(description, product_class, top_k=12):
    """Lexical retrieval over the full international catalog. No product whitelist."""
    q = tokenize(description) + tokenize(product_class.replace("_", " "))
    if not q:
        return []
    qset = set(q)
    scored = []
    # Prefer HS-6; fall back to HS-4 if the catalog row is only a heading.
    pool = HS6 or HS4 or HS_CATALOG
    for row in pool:
        tokens = tokenize(row["description"])
        if not tokens:
            continue
        tset = set(tokens)
        overlap = len(qset & tset)
        if overlap == 0:
            continue
        # light length prior: more specific 6-digit beats a vague heading
        bonus = 1 if len(row["hs_code"]) == 6 else 0
        scored.append((overlap + bonus, row))
    scored.sort(key=lambda x: x[0], reverse=True)
    # de-dup by hs6
    seen = set()
    out = []
    for _, row in scored:
        if row["hs_code"] in seen:
            continue
        seen.add(row["hs_code"])
        out.append(row)
        if len(out) >= top_k:
            break
    return out


def national_candidates(hs6, jurisdiction):
    if not hs6 or not jurisdiction:
        return []
    hits = []
    for row in NATIONAL_BY_JUR.get(jurisdiction, []):
        row_hs6 = re.sub(r"\D", "", str(row.get("hs6") or ""))
        nat = str(row.get("national_code") or "")
        if row_hs6 == hs6 or re.sub(r"\D", "", nat).startswith(hs6):
            hits.append(row)
    return hits


CLASS_PROMPT = """Assign ONE product class from this closed list:
{classes}

Use the physical object and primary function. Ignore brand names and part numbers.

Product description: {desc}

Reply with ONLY the class token."""


SELECT_PROMPT = """You are a customs classification assistant for the Harmonized System.

You may ONLY choose an hs6 value that appears in CANDIDATES.
If no candidate clearly matches the product, set hs6 to null and needs_review true.
Never use a SKU, model, UPC, EAN, or invoice number as a tariff code.
Do not invent 8-digit or 10-digit national codes unless they appear in NATIONAL_CANDIDATES.

PRODUCT: {desc}
SKU (not a tariff code): {sku}
PRODUCT_CLASS: {pclass}
ORIGIN COUNTRY: {origin}
DESTINATION COUNTRY: {dest_country}
DESTINATION ISO: {dest_iso}

CANDIDATES (international HS headings retrieved from the official-style catalog):
{cands}

NATIONAL_CANDIDATES for {dest_iso} (may be empty):
{nats}

Return ONLY JSON:
{{
  "hs6": "6 digits or null",
  "destination_national_code": "national code from NATIONAL_CANDIDATES or null",
  "gri_path": "short reason citing a candidate title",
  "confidence": "HIGH|MEDIUM|LOW",
  "needs_review": true or false
}}
"""


def format_hs_candidates(rows):
    lines = []
    for r in rows:
        parent = HS_BY_CODE.get(r["hs_code"][:4], {})
        lines.append(
            f"- {r['hs_code']} | {r['description']} | parent={parent.get('description','')}"
        )
    return "\n".join(lines) if lines else "(none)"


def format_national(rows):
    if not rows:
        return "(none — do not invent a national extension)"
    lines = []
    for r in rows:
        lines.append(
            f"- {r.get('national_code')} | hs6={r.get('hs6')} | {r.get('title') or r.get('legal_text')}"
        )
    return "\n".join(lines)


def validate_selection(result, sku, candidates, national_rows):
    hs6 = result.get("hs6")
    nat = result.get("destination_national_code")
    if isinstance(hs6, str) and hs6.lower() in {"null", "none", ""}:
        hs6 = None
    if isinstance(nat, str) and nat.lower() in {"null", "none", ""}:
        nat = None
    hs6_digits = re.sub(r"\D", "", str(hs6 or ""))
    nat_digits = re.sub(r"\D", "", str(nat or ""))
    sku_digits = re.sub(r"\D", "", str(sku or ""))
    cand_codes = {r["hs_code"] for r in candidates}

    if hs6_digits and hs6_digits == sku_digits:
        return None, None, True, "rejected: code equals SKU"
    if hs6_digits and hs6_digits not in cand_codes:
        return None, None, True, "rejected: hs6 not in retrieved candidates"
    if hs6_digits and hs6_digits not in SCHEDULE_HS6 and len(hs6_digits) == 6:
        # allow HS-4 candidate promoted only if retrieved as 4-digit
        if hs6_digits not in cand_codes:
            return None, None, True, "rejected: hs6 not in catalog"
    if len(hs6_digits) not in (0, 4, 6):
        return None, None, True, "rejected: not an HS-4/HS-6"
    allowed_nat = {re.sub(r"\D", "", str(r.get("national_code") or "")) for r in national_rows}
    if nat_digits and nat_digits not in allowed_nat:
        nat = None
        return hs6_digits or None, None, True, "national code dropped; not in destination overlay"
    if nat_digits and hs6_digits and not nat_digits.startswith(hs6_digits):
        return hs6_digits, None, True, "national code does not extend hs6"
    pretty_nat = None
    if nat_digits:
        for r in national_rows:
            if re.sub(r"\D", "", str(r.get("national_code") or "")) == nat_digits:
                pretty_nat = r.get("national_code")
                break
    return (hs6_digits or None), pretty_nat, False, "ok"


def classify_item(item, order_level):
    desc = item.get("product_description") or item.get("simplified_description") or ""
    sku = item.get("sku_code")
    origin = (
        item.get("country_of_manufacture")
        or order_level.get("shipper_country")
        or None
    )
    dest_country = order_level.get("receiver_country") or None
    dest_iso = iso2_from_country(dest_country)

    rule_class = guess_product_class(desc)
    try:
        llm_class = qwen_generate(
            [
                {"role": "system", "content": "Reply with one class token only."},
                {
                    "role": "user",
                    "content": CLASS_PROMPT.format(
                        classes=", ".join(PRODUCT_CLASSES),
                        desc=desc,
                    ),
                },
            ],
            max_new_tokens=16,
            temperature=0.0,
        )
        token = re.sub(r"[^a-z_]", "", (llm_class or "").lower().split()[0] if llm_class else "")
        product_class = token if token in PRODUCT_CLASSES else rule_class
    except Exception:
        product_class = rule_class

    chunks = retrieve_hs(desc, product_class, top_k=12)
    # If retrieval is empty, do not invent.
    nats = national_candidates(chunks[0]["hs_code"] if chunks else None, dest_iso)
    # also collect national rows for every retrieved hs6
    more = []
    for ch in chunks:
        more.extend(national_candidates(ch["hs_code"], dest_iso))
    # unique
    seen_n = set()
    nats = []
    for row in more:
        k = str(row.get("national_code"))
        if k not in seen_n:
            seen_n.add(k)
            nats.append(row)

    if not desc.strip():
        return {
            "product_class": "unknown",
            "hs6_code": None,
            "destination_tariff_code": None,
            "hs_code": None,
            "hts_destination_country": dest_country,
            "hts_destination_iso": dest_iso,
            "hts_origin_country": origin,
            "hts_extraction_source": "skipped: empty description",
            "classification_reason": "no product description",
            "confidence": "LOW",
            "needs_review": True,
            "validation_note": "empty description",
            "candidate_hs": [],
        }

    raw = qwen_generate(
        [
            {
                "role": "system",
                "content": "You may only output an hs6 that appears in CANDIDATES. Never guess.",
            },
            {
                "role": "user",
                "content": SELECT_PROMPT.format(
                    desc=desc,
                    sku=sku,
                    pclass=product_class,
                    origin=origin,
                    dest_country=dest_country,
                    dest_iso=dest_iso,
                    cands=format_hs_candidates(chunks),
                    nats=format_national(nats),
                ),
            },
        ],
        max_new_tokens=220,
        temperature=0.0,
    )
    try:
        selection = parse_json_maybe(raw)
    except Exception:
        selection = {
            "hs6": None,
            "destination_national_code": None,
            "gri_path": "parse_failed",
            "confidence": "LOW",
            "needs_review": True,
        }

    hs6, nat, forced_review, why = validate_selection(selection, sku, chunks, nats)
    needs_review = bool(selection.get("needs_review")) or forced_review or hs6 is None
    if dest_iso is None:
        needs_review = True
        why = (why + "; destination country not identified on invoice").strip("; ")
    if nat is None:
        needs_review = True  # national filing code unknown without overlay
    confidence = selection.get("confidence") or "LOW"
    if needs_review and confidence == "HIGH":
        confidence = "MEDIUM"

    return {
        "product_class": product_class,
        "hs6_code": hs6,
        "destination_tariff_code": nat,
        "hs_code": nat or hs6,
        "hts_destination_country": dest_country,
        "hts_destination_iso": dest_iso,
        "hts_origin_country": origin,
        "hts_extraction_source": "international HS catalog RAG + constrained Qwen",
        "classification_reason": selection.get("gri_path") or why,
        "confidence": confidence,
        "needs_review": needs_review,
        "validation_note": why,
        "candidate_hs": [c["hs_code"] for c in chunks],
    }


print("Generic classifier ready.")

Generic classifier ready.


In [ ]:
SELECT_PROMPT = """You are a customs classification assistant for the Harmonized System.

You may ONLY choose an hs6 value that appears in CANDIDATES.
If no candidate clearly matches the product, set hs6 to null and needs_review true.
Never use a SKU, model, UPC, EAN, or invoice number as a tariff code.
Do not invent 8-digit or 10-digit national codes unless they appear in NATIONAL_CANDIDATES.

PRODUCT: {desc}
SKU (not a tariff code): {sku}
PRODUCT_CLASS: {pclass}
ORIGIN COUNTRY: {origin} (ISO: {origin_iso})
DESTINATION COUNTRY: {dest_country} (ISO: {dest_iso})

CANDIDATES (international HS headings retrieved from the official-style catalog):
{cands}

ORIGIN_NATIONAL_CANDIDATES for {origin_iso} (may be empty):
{nats_orig}

DESTINATION_NATIONAL_CANDIDATES for {dest_iso} (may be empty):
{nats_dest}

Return ONLY JSON:
{{
  "hs6": "6 digits or null",
  "origin_national_code": "national code from ORIGIN_NATIONAL_CANDIDATES or null",
  "destination_national_code": "national code from DESTINATION_NATIONAL_CANDIDATES or null",
  "gri_path": "short reason citing a candidate title",
  "confidence": "HIGH|MEDIUM|LOW",
  "needs_review": true or false
}}
"""

def validate_selection(result, sku, candidates, dest_national_rows, orig_national_rows):
    hs6 = result.get("hs6")
    dest_nat = result.get("destination_national_code")
    orig_nat = result.get("origin_national_code")

    if isinstance(hs6, str) and hs6.lower() in {"null", "none", ""}:
        hs6 = None
    if isinstance(dest_nat, str) and dest_nat.lower() in {"null", "none", ""}:
        dest_nat = None
    if isinstance(orig_nat, str) and orig_nat.lower() in {"null", "none", ""}:
        orig_nat = None

    hs6_digits = re.sub(r"\D", "", str(hs6 or ""))
    dest_nat_digits = re.sub(r"\D", "", str(dest_nat or ""))
    orig_nat_digits = re.sub(r"\D", "", str(orig_nat or ""))
    sku_digits = re.sub(r"\D", "", str(sku or ""))
    cand_codes = {r["hs_code"] for r in candidates}

    if hs6_digits and hs6_digits == sku_digits:
        return None, None, None, True, "rejected: code equals SKU"
    if hs6_digits and hs6_digits not in cand_codes:
        return None, None, None, True, "rejected: hs6 not in retrieved candidates"
    if hs6_digits and hs6_digits not in SCHEDULE_HS6 and len(hs6_digits) == 6:
        if hs6_digits not in cand_codes:
            return None, None, None, True, "rejected: hs6 not in catalog"
    if len(hs6_digits) not in (0, 4, 6):
        return None, None, None, True, "rejected: not an HS-4/HS-6"

    allowed_dest_nat = {re.sub(r"\D", "", str(r.get("national_code") or "")) for r in dest_national_rows}
    if dest_nat_digits and dest_nat_digits not in allowed_dest_nat:
        dest_nat = None
    if dest_nat_digits and hs6_digits and not dest_nat_digits.startswith(hs6_digits):
        dest_nat = None

    allowed_orig_nat = {re.sub(r"\D", "", str(r.get("national_code") or "")) for r in orig_national_rows}
    if orig_nat_digits and orig_nat_digits not in allowed_orig_nat:
        orig_nat = None
    if orig_nat_digits and hs6_digits and not orig_nat_digits.startswith(hs6_digits):
        orig_nat = None

    pretty_dest_nat = None
    if dest_nat_digits:
        for r in dest_national_rows:
            if re.sub(r"\D", "", str(r.get("national_code") or "")) == dest_nat_digits:
                pretty_dest_nat = r.get("national_code")
                break

    pretty_orig_nat = None
    if orig_nat_digits:
        for r in orig_national_rows:
            if re.sub(r"\D", "", str(r.get("national_code") or "")) == orig_nat_digits:
                pretty_orig_nat = r.get("national_code")
                break

    return (hs6_digits or None), pretty_dest_nat, pretty_orig_nat, False, "ok"

def classify_item(item, order_level):
    desc = item.get("product_description") or item.get("simplified_description") or ""
    sku = item.get("sku_code")
    origin = (
        item.get("country_of_manufacture")
        or order_level.get("shipper_country")
        or None
    )
    dest_country = order_level.get("receiver_country") or None
    dest_iso = iso2_from_country(dest_country)
    origin_iso = iso2_from_country(origin)

    rule_class = guess_product_class(desc)
    try:
        llm_class = qwen_generate(
            [
                {"role": "system", "content": "Reply with one class token only."},
                {
                    "role": "user",
                    "content": CLASS_PROMPT.format(
                        classes=", ".join(PRODUCT_CLASSES),
                        desc=desc,
                    ),
                },
            ],
            max_new_tokens=16,
            temperature=0.0,
        )
        token = re.sub(r"[^a-z_]", "", (llm_class or "").lower().split()[0] if llm_class else "")
        product_class = token if token in PRODUCT_CLASSES else rule_class
    except Exception:
        product_class = rule_class

    chunks = retrieve_hs(desc, product_class, top_k=12)

    more_dest = []
    more_orig = []
    for ch in chunks:
        more_dest.extend(national_candidates(ch["hs_code"], dest_iso))
        more_orig.extend(national_candidates(ch["hs_code"], origin_iso))

    def unique_nats(more):
        seen = set()
        out = []
        for r in more:
            k = str(r.get("national_code"))
            if k not in seen:
                seen.add(k)
                out.append(r)
        return out

    nats_dest = unique_nats(more_dest)
    nats_orig = unique_nats(more_orig)

    if not desc.strip():
        return {
            "product_class": "unknown",
            "hs6_code": None,
            "manufacturing_country_hts_code": None,
            "destination_country_hts_code": None,
            "destination_tariff_code": None,
            "hs_code": None,
            "hts_destination_country": dest_country,
            "hts_destination_iso": dest_iso,
            "hts_origin_country": origin,
            "hts_extraction_source": "skipped: empty description",
            "classification_reason": "no product description",
            "confidence": "LOW",
            "needs_review": True,
            "validation_note": "empty description",
            "candidate_hs": [],
        }

    raw = qwen_generate(
        [
            {
                "role": "system",
                "content": "You may only output an hs6 that appears in CANDIDATES. Never guess.",
            },
            {
                "role": "user",
                "content": SELECT_PROMPT.format(
                    desc=desc,
                    sku=sku,
                    pclass=product_class,
                    origin=origin,
                    origin_iso=origin_iso,
                    dest_country=dest_country,
                    dest_iso=dest_iso,
                    cands=format_hs_candidates(chunks),
                    nats_dest=format_national(nats_dest),
                    nats_orig=format_national(nats_orig),
                ),
            },
        ],
        max_new_tokens=220,
        temperature=0.0,
    )
    try:
        selection = parse_json_maybe(raw)
    except Exception:
        selection = {
            "hs6": None,
            "origin_national_code": None,
            "destination_national_code": None,
            "gri_path": "parse_failed",
            "confidence": "LOW",
            "needs_review": True,
        }

    hs6, dest_nat, orig_nat, forced_review, why = validate_selection(selection, sku, chunks, nats_dest, nats_orig)
    needs_review = bool(selection.get("needs_review")) or forced_review or hs6 is None
    if dest_iso is None:
        needs_review = True
        why = (why + "; destination country not identified on invoice").strip("; ")
    if dest_nat is None:
        needs_review = True

    confidence = selection.get("confidence") or "LOW"
    if needs_review and confidence == "HIGH":
        confidence = "MEDIUM"

    return {
        "product_class": product_class,
        "hs6_code": hs6,
        "manufacturing_country_hts_code": orig_nat,
        "destination_country_hts_code": dest_nat,
        "destination_tariff_code": dest_nat,
        "hs_code": dest_nat,
        "hts_destination_country": dest_country,
        "hts_destination_iso": dest_iso,
        "hts_origin_country": origin,
        "hts_extraction_source": "international HS catalog RAG + constrained Qwen",
        "classification_reason": selection.get("gri_path") or why,
        "confidence": confidence,
        "needs_review": needs_review,
        "validation_note": why,
        "candidate_hs": [c["hs_code"] for c in chunks],
    }

In [ ]:
invoice = parsed_json["invoices"][0]
order_level = invoice.get("order_level", {})
items = invoice.get("item_level") or []

print("Invoice destination:", order_level.get("receiver_country"),
      "ISO:", iso2_from_country(order_level.get("receiver_country")))
print("Invoice origin/shipper:", order_level.get("shipper_country"))
print(f"Classifying {len(items)} line item(s) against the international HS catalog...")

for i, item in enumerate(items, 1):
    print(f"\n[{i}/{len(items)}] sku={item.get('sku_code')} | {item.get('product_description')}")
    result = classify_item(item, order_level)
    item.update(result)
    print(
        f"  class={result['product_class']} hs6={result['hs6_code']} "
        f"national={result['destination_tariff_code']} "
        f"dest={result['hts_destination_iso']} "
        f"conf={result['confidence']} review={result['needs_review']}"
    )
    print(f"  reason={result['classification_reason']}")

final_path = "/content/output/final_classification.json"
os.makedirs("/content/output", exist_ok=True)
with open(final_path, "w", encoding="utf-8") as f:
    json.dump(parsed_json, f, indent=2)
print(f"\nSaved {final_path}")

Invoice destination: Australia ISO: AU
Invoice origin/shipper: None
Classifying 29 line item(s) against the international HS catalog...

[1/29] sku=DFS00031 | DF Master Card 220V DF800
  class=unknown hs6=844831 national=None dest=AU conf=MEDIUM review=True
  reason=Machines, card clothing; parts and accessories

[2/29] sku=DFS00079 | DF Sorting Gate Control Box IR DF101
  class=control_board_or_panel hs6=None national=None dest=AU conf=LOW review=True
  reason=Control board or panel does not match any provided candidates.

[3/29] sku=IDU00034 | LD ID Unit IDU510
  class=sensor_or_measuring_instrument hs6=None national=None dest=AU conf=LOW review=True
  reason=None of the provided candidates match the product description.

[4/29] sku=CBX00030 | MC200 Milking/Rinsing Control
  class=machine_part hs6=None national=None dest=AU conf=LOW review=True
  reason=No suitable candidate found for MC200 Milking/Rinsing Control

[5/29] sku=CBX00039 | MC200/ED Connection Box CB200A
  class=control_

In [ ]:
from IPython.display import JSON, display

rows = []
for item in parsed_json["invoices"][0].get("item_level", []):
    rows.append(
        {
            "sku": item.get("sku_code"),
            "description": item.get("product_description"),
            "class": item.get("product_class"),
            "hs6": item.get("hs6_code"),
            "au_code": item.get("destination_tariff_code"),
            "confidence": item.get("confidence"),
            "needs_review": item.get("needs_review"),
            "reason": item.get("classification_reason"),
        }
    )

print("Line-item classification summary")
for r in rows:
    print(f"{r['sku']:<12} {str(r['hs6']):<8} {str(r['au_code']):<14} {r['class']:<22} {r['confidence']:<6} review={r['needs_review']}")
    print(f"             {r['description']}")

display(JSON(parsed_json))

Line-item classification summary
DFS00031     844831   None           unknown                MEDIUM review=True
             DF Master Card 220V DF800
DFS00079     None     None           control_board_or_panel LOW    review=True
             DF Sorting Gate Control Box IR DF101
IDU00034     None     None           sensor_or_measuring_instrument LOW    review=True
             LD ID Unit IDU510
CBX00030     None     None           machine_part           LOW    review=True
             MC200 Milking/Rinsing Control
CBX00039     None     None           control_board_or_panel LOW    review=True
             MC200/ED Connection Box CB200A
FFM00053     020990   None           food_or_beverage       MEDIUM review=True
             Fat Sampler FFS Cow/sheep
FFS00013     None     None           sensor_or_measuring_instrument LOW    review=True
             MC200 FF Sensor FFS30C 24V Green 3M Cable
FFS00027     None     None           sensor_or_measuring_instrument LOW    review=True
          

<IPython.core.display.JSON object>

### Retrieve and display the full HTS code tree for a specific country

This section demonstrates how to retrieve all HTS codes for a chosen country and construct a hierarchical tree structure from the cached data.

In [ ]:
# Cache for storing full HTS data for each country
_full_hts_data_by_country = {}

def _get_full_hts_data_for_country(country_code, hs_collection):
    """Fetches and caches all HTS data for a given country from Weaviate."""
    if country_code in _full_hts_data_by_country:
        return _full_hts_data_by_country[country_code]

    print(f"Fetching all HTS data for country code {country_code} from Weaviate...")
    all_hts_data = []
    try:
        # Fetch all objects for the given country code
        response = hs_collection.query.fetch_objects(
            filters=weaviate.classes.query.Filter.by_property("country_code").equal(country_code),
            return_properties=["code", "description", "indent"],
            limit=10000 # Adjust limit if a country has more than 10k HTS codes
        )

        for obj in response.objects:
            props = obj.properties
            all_hts_data.append({
                "code": props.get("code"),
                "description": props.get("description"),
                "indent": props.get("indent"),
            })
    except Exception as e:
        print(f"Error fetching full HTS data for {country_code}: {e}")
        return {"flat_list": [], "tree": []}

    # Sort the flat list for consistent tree building later
    sorted_flat_list = sorted(all_hts_data, key=lambda x: x.get('code', ''))

    # Store in cache
    _full_hts_data_by_country[country_code] = {"flat_list": sorted_flat_list}

    print(f"Fetched {len(sorted_flat_list)} HTS codes for {country_code}.")
    return _full_hts_data_by_country[country_code]

print("_get_full_hts_data_for_country helper function ready.")

_get_full_hts_data_for_country helper function ready.


In [ ]:
def build_hts_tree_for_country(country_code, hs_collection):
    """Retrieves all HTS codes for a country and builds a hierarchical tree structure."""
    print(f"Building HTS tree for country code: {country_code}")
    country_data = _get_full_hts_data_for_country(country_code, hs_collection)
    all_hts_flat = sorted(country_data["flat_list"], key=lambda x: x.get('code', ''))

    tree = []
    # A dictionary to keep track of the last node at each indent level
    # This is crucial for correctly linking children to their parents.
    level_stack = {}

    for item in all_hts_flat:
        code = item.get('code')
        description = item.get('description')
        indent_str = item.get('indent')

        if not code or not description or not indent_str:
            continue

        try:
            indent = int(indent_str)
        except ValueError:
            continue

        node = {
            'code': code,
            'description': description,
            'indent': indent,
            'children': []
        }

        if indent == 1: # Top-level item
            tree.append(node)
            level_stack = {1: node} # Reset stack for new top-level branch
        else:
            # Find the parent at the previous indent level
            parent_indent = indent - 1
            if parent_indent in level_stack:
                # Append to the children of the immediate parent
                level_stack[parent_indent]['children'].append(node)
            else:
                # If no immediate parent found, append to the highest available parent or top level
                # This handles cases where a code might be missing intermediate parents in the flat list.
                # However, for well-formed HTS data, the immediate parent should usually be present.
                found_parent = False
                for lvl in sorted(level_stack.keys(), reverse=True):
                    if lvl < indent:
                        level_stack[lvl]['children'].append(node)
                        found_parent = True
                        break
                if not found_parent:
                    tree.append(node) # Fallback: add to top level if no parent found in stack

            level_stack[indent] = node # Update the stack with the current node at its indent level

            # Clean up lower levels in the stack, as new nodes at this level or above imply their branches are closed
            for lvl in list(level_stack.keys()):
                if lvl > indent:
                    del level_stack[lvl]

    return tree


def print_tree(nodes, level=0):
    """Helper function to pretty-print the HTS tree."""
    indent_str = "  " * level
    for node in nodes:
        print(f"{indent_str}- {node['code']} {node['description']} (Indent: {node['indent']})")
        if node['children']:
            print_tree(node['children'], level + 1)

print("HTS tree building function ready.")

HTS tree building function ready.


In [ ]:
# Example usage:

# Ensure hs_category_collection is available
if 'weaviate_client' not in globals() or not weaviate_client:
    raise RuntimeError("Weaviate client not connected. Please run the Weaviate connection cells first.")
h_collection = weaviate_client.collections.get("Hs_category")

# Choose a country code to display its HTS tree
# This will trigger the caching of all HTS codes for this country if not already cached
country_to_display = "107" # Changed to '107' for Israel

full_hts_tree = build_hts_tree_for_country(country_to_display, h_collection)

# --- New code to save and display JSON ---
import json
from IPython.display import JSON, display
import os

output_json_path = f"/content/output/hts_tree_country_{country_to_display}.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(full_hts_tree, f, indent=2)

print(f"\nFull HTS Tree for Country Code {country_to_display} saved to: {output_json_path}")
print("\n--- Displaying Full HTS Tree JSON ---")
display(JSON(full_hts_tree))
print("-------------------------------------")


Building HTS tree for country code: 107
Fetching all HTS data for country code 107 from Weaviate...
Fetched 0 HTS codes for 107.

Full HTS Tree for Country Code 107 saved to: /content/output/hts_tree_country_107.json

--- Displaying Full HTS Tree JSON ---


<IPython.core.display.JSON object>

-------------------------------------


## STEP 8 — Optional full pipeline function

Use this when you want one call per PDF after models are loaded.


### Debugging HTS Classification for 'DF Master Card 220V'

In [ ]:
import json
import copy
import os

# Ensure parsed_json is loaded, as its state might be lost in the kernel
extracted_invoice_path = "/content/output/extracted_invoice.json"
if os.path.exists(extracted_invoice_path):
    with open(extracted_invoice_path, 'r', encoding='utf-8') as f:
        parsed_json = json.load(f)
    print(f"Loaded parsed_json from {extracted_invoice_path}")
else:
    raise FileNotFoundError(f"'{extracted_invoice_path}' not found. Please run previous cells to generate it.")

# Find the specific item 'DF Master Card 220V'
item_to_debug = None
order_level_debug = parsed_json["invoices"][0].get("order_level", {})
for item in parsed_json["invoices"][0].get("item_level", []):
    if item.get('sku_code') == 'DFS00031':  # Assuming DFS00031 is the SKU for 'DF Master Card 220V'
        item_to_debug = copy.deepcopy(item)
        break

if item_to_debug:
    print(f"Debugging classification for: {item_to_debug.get('product_description')} (SKU: {item_to_debug.get('sku_code')})")

    # Check if classify_item is defined
    if 'classify_item' not in globals():
        print("\n--- ERROR: `classify_item` function not defined ---")
        print("Please ensure cell `21a6153b` (containing classify_item definition) has been executed.\n")
    else:
        # Capture current print functions to redirect output from qwen_generate
        original_qwen_generate = globals().get('qwen_generate')

        # Override qwen_generate to capture prompts and responses
        def debug_qwen_generate(messages, max_new_tokens=8000, temperature=0.0):
            print("\n--- Qwen Prompt ---")
            print(messages[1]['content'])
            print("-------------------")
            response = original_qwen_generate(messages, max_new_tokens, temperature)
            print("\n--- Qwen Raw Response ---")
            print(response)
            print("-----------------------")
            return response

        globals()['qwen_generate'] = debug_qwen_generate

        try:
            # Re-run classify_item for the specific item
            result = classify_item(item_to_debug, order_level_debug)
            item_to_debug.update(result)

            print("\n--- Classification Result for DF Master Card 220V ---")
            print(f"Product Class: {result.get('product_class')}")
            print(f"HS6 Code: {result.get('hs6_code')}")
            print(f"Destination National Code: {result.get('destination_national_code')}")
            print(f"Confidence: {result.get('confidence')}")
            print(f"Needs Review: {result.get('needs_review')}")
            print(f"Classification Reason: {result.get('classification_reason')}")
            print(f"Validation Note: {result.get('validation_note')}")
            print(f"Candidate HS codes considered: {result.get('candidate_hs')}")

            print("\n--- Full Classification JSON for DF Master Card 220V ---")
            from IPython.display import JSON, display
            display(JSON(item_to_debug))

        finally:
            # Restore original qwen_generate
            globals()['qwen_generate'] = original_qwen_generate

else:
    print("Item 'DF Master Card 220V' (SKU DFS00031) not found in parsed_json.")

Loaded parsed_json from /content/output/extracted_invoice.json
Debugging classification for: DF Master Card 220V (SKU: DFS00031)

--- ERROR: `classify_item` function not defined ---
Please ensure cell `21a6153b` (containing classify_item definition) has been executed.



In [ ]:
def hts_classification_pipeline(pdf_file_path):
    """OCR one unknown PDF -> extract JSON -> generic HS RAG. No vendor priors."""
    global cleaned_text, parsed_json
    if not os.path.exists(pdf_file_path):
        raise FileNotFoundError(pdf_file_path)

    out_dir = "/content/output/pipeline_pdf"
    os.makedirs(out_dir, exist_ok=True)
    paths = pdf_to_images(pdf_file_path, dpi=350)

    old_out, old_err = sys.stdout, sys.stderr
    devnull = open(os.devnull, "w")
    try:
        sys.stdout = sys.stderr = devnull
        ocr_model.infer_multi(
            ocr_tokenizer,
            prompt="<image>Multi page parsing.",
            image_files=paths,
            output_path=out_dir,
            image_size=1024,
            max_length=32768,
            no_repeat_ngram_size=35,
            ngram_window=1024,
            save_results=True,
        )
    finally:
        sys.stdout, sys.stderr = old_out, old_err
        devnull.close()

    raw = ""
    md_path = os.path.join(out_dir, "result.md")
    if os.path.exists(md_path):
        with open(md_path, "r", encoding="utf-8") as f:
            raw = f.read()
    cleaned_text = re.sub(r"<\|det\|>[^<]*?<\|/det\|>", "", raw, flags=re.DOTALL)
    cleaned_text = re.sub(r"<[^>]*>", "", cleaned_text, flags=re.DOTALL).strip()

    raw_json = qwen_generate(
        [
            {"role": "system", "content": EXTRACT_SYSTEM_PROMPT},
            {"role": "user", "content": f"OCR Text:\n\n{cleaned_text}"},
        ],
        max_new_tokens=10000,
        temperature=0.0,
    )
    parsed_json = parse_json_maybe(raw_json)
    inv = parsed_json["invoices"][0]
    for item in inv.get("item_level", []):
        item["hs_code"] = None
        item["manufacturing_country_hts_code"] = None
        item["destination_country_hts_code"] = None
        item.update(classify_item(item, inv.get("order_level", {})))
    return parsed_json


print("hts_classification_pipeline(pdf_path) is ready.")

## Cleanup (optional)

Run only when you need VRAM back.


In [ ]:
import gc

for name in ("ocr_model", "Qmodel", "ocr_tokenizer", "Qtokenizer"):
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Models released.")

Models released.


## Design constraints for thousands of unknown PDFs

| Temptation | Why it is rejected |
|---|---|
| Hardcode SKUs / model lines | Next PDF is a different catalog |
| Default destination = Australia | Next PDF may be US, DE, IN, … |
| Google “HS code for {description}” | SEO pages are not a schedule |
| Always emit 10-digit HTSUS | Wrong instrument outside the US |

**What is static (allowed)**
- International HS-2/4/6 titles (shared legal language).
- A country-name → ISO2 map so `receiver_country` can select an overlay file.
- Generic function words (`valve`, `cable`, `gasket`) as a weak prior only. Qwen class still wins if it returns a valid enum token.

**What is dynamic (from each PDF)**
- Line items and descriptions.
- `receiver_country` → destination ISO.
- `shipper_country` / `country_of_manufacture` → origin (shown, not used to pick the import code).

**National 8/10-digit codes**
Drop official files at `/content/tariff_index_national.json` keyed by `jurisdiction` (`US`, `AU`, `GB`, …).  
If that file is missing for the destination on the invoice, `destination_tariff_code` stays `null` and `needs_review=true`. That is correct behavior, not a failure.

Batch use:

```python
for pdf in list_of_pdfs:
    result = hts_classification_pipeline(pdf)
    # persist result; route needs_review=true rows to a human queue
```
